# CSCI 6353 · Topic 8 — Monte Carlo Control

Learn the optimal policy in a 5×7 maze GridWorld (walls, reward −1 per move) with MC control: an ε-greedy Q-table agent, backward MC updates, and ε annealing.

- Cell 1: the maze GridWorld environment (walls in the move methods).
- Cell 2: `QAgent` — Q-table (5×7×4), ε-greedy action selection, MC `update_table`, `anneal_eps`, `show_table`.
- Cell 3: main loop (5,000 episodes) — prints the greedy action per cell.

*Dr. Dongchul Kim · Summer II 2026*


In [1]:
import random

class GridWorld():
    def __init__(self):
        self.x=2
        self.y=0

    def step(self, a):
        if a==0:
            self.move_left()
        elif a==1:
            self.move_up()
        elif a==2:
            self.move_right()
        elif a==3:
            self.move_down()

        reward = -1
        done = self.is_done()
        return (self.x, self.y), reward, done

    def move_left(self):
        if self.y==0:
            pass
        elif self.y==3 and self.x in [0,1,2]:
            pass
        elif self.y==5 and self.x in [2,3,4]:
            pass
        else:
            self.y -= 1

    def move_right(self):
        if self.y==1 and self.x in [0,1,2]:
            pass
        elif self.y==3 and self.x in [2,3,4]:
            pass
        elif self.y==6:
            pass
        else:
            self.y += 1

    def move_up(self):
        if self.x==0:
            pass
        elif self.x==3 and self.y==2:
            pass
        else:
            self.x -= 1

    def move_down(self):
        if self.x==4:
            pass
        elif self.x==1 and self.y==4:
            pass
        else:
            self.x+=1

    def is_done(self):
        if self.x==4 and self.y==6:
            return True
        else:
            return False

    def reset(self):
        self.x = 0
        self.y = 0
        return (self.x, self.y)



In [4]:
import numpy as np

class QAgent():
    def __init__(self):
        self.q_table = np.zeros((5, 7, 4))
        self.eps = 0.9
        self.alpha = 0.01
        self.gamma = 0.9

    def select_action(self, s): # eps-greedy
        x, y = s
        coin = random.random()
        if coin < self.eps: # random action
            action = random.randint(0,3)
        else:
            action_val = self.q_table[x,y,:]
            action = np.argmax(action_val)
        return action

    def update_table(self, history):
        # Given a single episode, the q-table's values are updated accordingly.
        cum_reward = 0
        for transition in history[::-1]: # from the last transition
            s, a, r, s_prime = transition
            x,y = s
            self.q_table[x,y,a] = self.q_table[x,y,a] + self.alpha * (cum_reward - self.q_table[x,y,a])
            cum_reward = r + self.gamma * cum_reward

    def anneal_eps(self):
        self.eps -= 0.03
        self.eps = max(self.eps, 0.1)

    def show_table(self):
        q_lst = self.q_table.tolist()
        data = np.zeros((5,7))
        for row_idx in range(len(q_lst)):
            row = q_lst[row_idx]
            for col_idx in range(len(row)):
                col = row[col_idx]
                action = np.argmax(col)
                data[row_idx, col_idx] = action
        print(data)



In [ ]:
def main():
    env = GridWorld()
    agent = QAgent()

    for n_epi in range(5000): # 1000 episodes
        done = False
        history = []

        s = env.reset()
        while not done: # single episode
            a = agent.select_action(s)
            s_prime, r, done = env.step(a)
            history.append((s, a, r, s_prime))
            s = s_prime
        agent.update_table(history) # update table
        agent.anneal_eps()

    agent.show_table()

if __name__ == '__main__':
    main()

[[3. 3. 0. 2. 3. 3. 0.]
 [3. 3. 0. 2. 2. 3. 3.]
 [2. 3. 0. 1. 0. 3. 3.]
 [1. 2. 2. 1. 0. 3. 3.]
 [2. 2. 2. 1. 0. 2. 0.]]


0 → Left

1 → Up

2 → Right

3 → Down